# Fase 3 · CoALA
Trabalho: 12 mensagens; episódios: FastEmbed multilíngue; preferências explícitas; procedimento: verificação. Mem0 e Letta administram memória com abstrações próprias; aqui o SQLite conserva identidade e versão vetorial.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.memory import (FastEmbedder, Historico, nova_sessao, salvar_episodio,
                          atualizar_episodio, recuperar, salvar_preferencia, preferencias)
from langchain_core.messages import HumanMessage, AIMessage
caminho = Path('storage/notebooks/memoria.sqlite3')
sessao = nova_sessao(caminho, 'aluna', 'demo')
historico = Historico(caminho, 'aluna', sessao)
historico.add_messages([HumanMessage(content='Quero duas peças'), AIMessage(content='Qual prazo?')])
historico.add_messages([HumanMessage(content='Sete dias'), AIMessage(content='Entendido')])
salvar_preferencia(caminho, 'aluna', 'fornecedor', 'nacional', explicita=True)
emb = FastEmbedder()
eid = salvar_episodio(caminho, 'aluna', sessao, 'prefere fornecedor nacional', emb)
print([m.content for m in historico.todos()], preferencias(caminho, 'aluna'))
print('antes', recuperar(caminho, 'aluna', 'fornecedor nacional', emb))
atualizar_episodio(caminho, 'aluna', eid, 'corrigiu: aceita fornecedor internacional', emb)
print('depois', recuperar(caminho, 'aluna', 'fornecedor internacional', emb))

/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['Quero duas peças', 'Qual prazo?', 'Sete dias', 'Entendido'] {'fornecedor': 'nacional'}
antes [{'id': 9, 'dataset_id': 'demo', 'texto': 'prefere fornecedor nacional', 'similaridade': np.float64(0.6676986243721618)}, {'id': 1, 'dataset_id': 'demo', 'texto': 'corrigiu: aceita fornecedor internacional', 'similaridade': np.float64(0.5507463600207149)}, {'id': 2, 'dataset_id': 'demo', 'texto': 'corrigiu: aceita fornecedor internacional', 'similaridade': np.float64(0.5507463600207149)}]
depois [{'id': 1, 'dataset_id': 'demo', 'texto': 'corrigiu: aceita fornecedor internacional', 'similaridade': np.float64(0.7076525577288841)}, {'id': 2, 'dataset_id': 'demo', 'texto': 'corrigiu: aceita fornecedor internacional', 'similaridade': np.float64(0.7076525577288841)}, {'id': 3, 'dataset_id': 'demo', 'texto': 'corrigiu: aceita fornecedor internacional', 'similaridade': np.float64(0.7076525577288841)}]


Preço e estoque guardados em episódios são históricos. O pedido atual prevalece.

## Mapa de memórias (CoALA)
Execução real local, sem modelo. Depois de dois turnos de compra e uma preferência, `mapa_memorias` mostra o que há em cada tipo de memória. A conversa equivalente com o modelo real está em `docs/rastros/fase3_memoria_llm.json`.

In [3]:
import json
from pedro.agent import criar_fluxo
from pedro.catalog import criar_cenario
from pedro.memory import mapa_memorias
cenario = criar_cenario(Path('storage/notebooks/catalogos'))
caminho13 = Path('storage/notebooks/mapa_de_memorias.sqlite3')
sessao13 = nova_sessao(caminho13, 'aluna', cenario.dataset_id)
local = criar_fluxo(cenario, caminho13)
for texto in ('Preciso de 5 unidades de PEC-01-0001 com prazo 7 dias', 'Agora que sejam 6 e mantenha o prazo'):
    print(texto, '->', local.invoke({'input': texto}, config={'configurable': {
        'user_id': 'aluna', 'session_id': sessao13}}).content.splitlines()[0])
salvar_preferencia(caminho13, 'aluna', 'fornecedor', 'nacionais', explicita=True)
salvar_episodio(caminho13, 'aluna', sessao13, 'pediu 6 unidades de PEC-01-0001 com prazo 7 dias', emb)
for memoria_coala, conteudo in mapa_memorias(caminho13, 'aluna', sessao13).items():
    print(memoria_coala, '->', json.dumps(conteudo, ensure_ascii=False))

Preciso de 5 unidades de PEC-01-0001 com prazo 7 dias -> CentroPeças: R$ 7.279,83 para 5 unidades; prazo de 2 dias.
Agora que sejam 6 e mantenha o prazo -> CentroPeças: R$ 8.718,75 para 6 unidades; prazo de 2 dias.
trabalho (curto prazo, por sessão) -> {"rascunho_de_compra": {"produto_id": "PEC-01-0001", "quantidade": 6, "prazo_max_dias": 7}, "mensagens_na_janela": 4, "mensagens_salvas": 4}
episódica (longo prazo, por usuário) -> {"episodios": 2, "sessoes_de_origem": 2, "recuperacao": "similaridade de cosseno, top 3, modelo sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
semântica (longo prazo) -> {"preferencias_do_usuario": {"fornecedor": "nacionais"}, "conhecimento_do_dominio": "catálogo SQLite do cenário e políticas em FAISS"}
procedimental (longo prazo) -> {"onde": "prompt PTCF, exemplos few-shot, regras de diálogo e código de verificação", "muda": "somente com uma nova versão do projeto"}


/tmp/ipykernel_556551/1326061929.py:8: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  local = criar_fluxo(cenario, caminho13)


## Chatbot com ConversationBufferMemory
**Duplo de teste:** `FakeListLLM` devolve respostas fixas. O real é `ConversationChain` com `ConversationBufferMemory`: o prompt do último turno é impresso para ver que ele leva a conversa inteira. A conversa com o modelo real está em `docs/rastros/fase3_memoria_llm.json`.

In [4]:
from langchain_core.language_models.fake import FakeListLLM
from pedro.legacy import criar_chatbot_buffer
chatbot, buffer = criar_chatbot_buffer(FakeListLLM(responses=[
    'Olá, Ana, de quantas unidades você precisa?', 'Anotado: 2 unidades. Orçamento e prazo?',
    'Ana, você pediu 2 termostatos, até R$ 900, em 5 dias.']))
for texto in ('Olá, sou a Ana. Preciso de um termostato para a minha geladeira.', 'Que sejam 2 unidades.',
              'Tenho até R$ 900 e preciso dele em 5 dias. Resuma a minha solicitação.'):
    print('Usuário:', texto)
    print('Pedro  :', chatbot.invoke({'input': texto})['response'])
    print('   caracteres na memória:', len(buffer.load_memory_variables({})['history']))
print('--- memória de buffer ---')
print(buffer.load_memory_variables({})['history'])

Usuário: Olá, sou a Ana. Preciso de um termostato para a minha geladeira.
Pedro  : Olá, Ana, de quantas unidades você precisa?
   caracteres na memória: 124
Usuário: Que sejam 2 unidades.
Pedro  : Anotado: 2 unidades. Orçamento e prazo?
   caracteres na memória: 202
Usuário: Tenho até R$ 900 e preciso dele em 5 dias. Resuma a minha solicitação.
Pedro  : Ana, você pediu 2 termostatos, até R$ 900, em 5 dias.
   caracteres na memória: 343
--- memória de buffer ---
Usuário: Olá, sou a Ana. Preciso de um termostato para a minha geladeira.
Pedro: Olá, Ana, de quantas unidades você precisa?
Usuário: Que sejam 2 unidades.
Pedro: Anotado: 2 unidades. Orçamento e prazo?
Usuário: Tenho até R$ 900 e preciso dele em 5 dias. Resuma a minha solicitação.
Pedro: Ana, você pediu 2 termostatos, até R$ 900, em 5 dias.


/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/src/pedro/legacy.py:122: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(human_prefix="Usuário", ai_prefix="Pedro")
/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic